In [ ]:
import time
from collections import deque


def in_mt(mt):
    mt = tuple(mt)
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i * 3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res


def get_successors(state):
    state = tuple(state)
    pos = state.index(0)
    r, c = pos // 3, pos % 3

    successors = []

    def swap_tuple(s, i, j):
        new_s = list(s)
        new_s[i], new_s[j] = new_s[j], new_s[i]
        return tuple(new_s)

    if c > 0:
        successors.append(("Trái", swap_tuple(state, pos, pos - 1)))

    if c < 2:
        successors.append(("Phải", swap_tuple(state, pos, pos + 1)))

    if r > 0:
        successors.append(("Lên", swap_tuple(state, pos, pos - 3)))

    if r < 2:
        successors.append(("Xuống", swap_tuple(state, pos, pos + 3)))

    return successors


def is_successor(s1, s2):
    s1 = tuple(s1)
    s2 = tuple(s2)

    for action, child in get_successors(s1):
        if child == s2:
            return True

    return False


def get_action(s1, s2):
    s1 = tuple(s1)
    s2 = tuple(s2)

    for action, child in get_successors(s1):
        if child == s2:
            return action

    return None


def bfs_distances(source, max_depth):
    source = tuple(source)

    dist = {source: 0}
    queue = deque([source])

    while queue:
        state = queue.popleft()

        if dist[state] >= max_depth:
            continue

        for action, child in get_successors(state):
            if child not in dist:
                dist[child] = dist[state] + 1
                queue.append(child)

    return dist


def build_domains(start_state, goal_state, k):
    start_state = tuple(start_state)
    goal_state = tuple(goal_state)

    if k == 0:
        if start_state == goal_state:
            return {"X0": {start_state}}
        else:
            return {"X0": set()}

    dist_from_start = bfs_distances(start_state, k)
    dist_from_goal = bfs_distances(goal_state, k)

    domains = {}

    for i in range(k + 1):
        var = f"X{i}"

        if i == 0:
            domains[var] = {start_state}

        elif i == k:
            domains[var] = {goal_state}

        else:
            domain_i = set()

            for state in dist_from_start:
                if state not in dist_from_goal:
                    continue

                d_start = dist_from_start[state]
                d_goal = dist_from_goal[state]

                can_reach_from_start = d_start <= i and (i - d_start) % 2 == 0
                can_reach_goal = d_goal <= (k - i) and ((k - i) - d_goal) % 2 == 0

                if can_reach_from_start and can_reach_goal:
                    domain_i.add(state)

            domains[var] = domain_i

    return domains


def constraint_ok(xi, vi, xj, vj):
    i = int(xi[1:])
    j = int(xj[1:])

    if j == i + 1:
        return is_successor(vi, vj)

    if j == i - 1:
        return is_successor(vj, vi)

    return False


def revise(domains, xi, xj):
    revised = False

    for vi in list(domains[xi]):
        has_support = False

        for vj in domains[xj]:
            if constraint_ok(xi, vi, xj, vj):
                has_support = True
                break

        if not has_support:
            domains[xi].remove(vi)
            revised = True

    return revised


def ac3(domains, k):
    queue = deque()

    for i in range(k):
        xi = f"X{i}"
        xj = f"X{i + 1}"

        queue.append((xi, xj))
        queue.append((xj, xi))

    neighbors = {}

    for i in range(k + 1):
        var = f"X{i}"
        neighbors[var] = []

        if i > 0:
            neighbors[var].append(f"X{i - 1}")

        if i < k:
            neighbors[var].append(f"X{i + 1}")

    while queue:
        xi, xj = queue.popleft()

        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False

            for xk in neighbors[xi]:
                if xk != xj:
                    queue.append((xk, xi))

    return True


def extract_solution(domains, k):
    assignment = [None] * (k + 1)

    assignment[0] = next(iter(domains["X0"]))

    def backtrack_step(i):
        if i == k:
            return assignment[i] in domains[f"X{i}"]

        current = assignment[i]
        next_var = f"X{i + 1}"

        for next_state in domains[next_var]:
            if is_successor(current, next_state):
                assignment[i + 1] = next_state

                result = backtrack_step(i + 1)
                if result:
                    return True

        assignment[i + 1] = None
        return False

    if backtrack_step(0):
        path = []

        for i in range(k):
            s1 = assignment[i]
            s2 = assignment[i + 1]
            action = get_action(s1, s2)
            path.append((action, s2))

        return path

    return None


def ac3_search(start_state, goal_state, limit):
    start_state = tuple(start_state)
    goal_state = tuple(goal_state)

    nodes_generated = 0

    for k in range(limit + 1):
        domains = build_domains(start_state, goal_state, k)

        nodes_generated += sum(len(domains[var]) for var in domains)

        if any(len(domains[var]) == 0 for var in domains):
            continue

        result = ac3(domains, k)

        if result:
            path = extract_solution(domains, k)

            if path is not None:
                return path, nodes_generated

    return None, nodes_generated